# Hands-On Module 1 Spotify Songs EDA

This notebook answers the full Hands-On Module 1 task using **Pandas** and **NumPy**. The goal is to understand what the data says about Spotify tracks, genres, popularity, and audio features.

## Stage 1  Setup and Initial Inspection

First, I load the Spotify Songs dataset directly from the given TidyTuesday URL. After loading, I check the shape, data types, missing values, and duplicate rows. These checks are important because bad data quality can silently affect all later analysis.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

DATA_URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"

df_raw = pd.read_csv(DATA_URL)
df_raw.head()

,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,playlist_subgenre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.748,0.916,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.726,0.815,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.675,0.931,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.718,0.930,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.650,0.833,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


In [2]:
print("Shape:", df_raw.shape)
print("Data types:")
display(df_raw.dtypes)

missing_table = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percent": df_raw.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

missing_table

Shape: (32833, 23)
Data types:


track_id                     object
track_name                   object
track_artist                 object
track_popularity              int64
track_album_id               object
track_album_name             object
track_album_release_date     object
playlist_name                object
playlist_id                  object
playlist_genre               object
playlist_subgenre            object
danceability                float64
energy                      float64
key                           int64
loudness                    float64
mode                          int64
speechiness                 float64
acousticness                float64
instrumentalness            float64
liveness                    float64
valence                     float64
tempo                       float64
duration_ms                   int64
dtype: object

,missing_count,missing_percent
track_name,5,0.015229
track_artist,5,0.015229
track_album_name,5,0.015229
track_id,0,0.000000
track_popularity,0,0.000000
track_album_id,0,0.000000
track_album_release_date,0,0.000000
playlist_name,0,0.000000
playlist_id,0,0.000000
playlist_genre,0,0.000000


### Handling missing columns

The rule from the module says that columns with more than 20% missing values should be dropped. I calculate this automatically below. If no column passes that threshold, no column will be dropped.

In [3]:
missing_threshold = 20
cols_to_drop = missing_table.index[missing_table["missing_percent"] > missing_threshold].tolist()

print("Columns dropped because missing values > 20%:", cols_to_drop)

df = df_raw.drop(columns=cols_to_drop).copy()
print("Shape after dropping high-missing columns:", df.shape)

Columns dropped because missing values > 20%: []
Shape after dropping high-missing columns: (32833, 23)


### Duplicate strategy

For exact duplicate rows, I remove them directly. For logical duplicates, I use `track_id` and `playlist_id` when both columns exist, because the same song can appear in several playlists, but the same song inside the same playlist should not need repeated identical records for this EDA. If the columns are not available, the notebook falls back to exact duplicate handling only.

In [4]:
exact_duplicates = df.duplicated().sum()
print("Exact duplicate rows:", exact_duplicates)

df = df.drop_duplicates().copy()

logical_keys = [col for col in ["track_id", "playlist_id"] if col in df.columns]
if len(logical_keys) == 2:
    logical_duplicates = df.duplicated(subset=logical_keys).sum()
    print(f"Logical duplicates using {logical_keys}:", logical_duplicates)
    df = df.drop_duplicates(subset=logical_keys, keep="first").copy()
else:
    print("Logical duplicate check skipped because track_id and/or playlist_id is not available.")

print("Final clean shape:", df.shape)

Exact duplicate rows: 0
Logical duplicates using ['track_id', 'playlist_id']: 582
Final clean shape: (32251, 23)


### Numeric summary table

Here I build a numeric summary table containing mean, median, standard deviation, and IQR. The IQR is calculated manually using NumPy percentiles, as required by the module.

In [5]:
def build_numeric_summary(data: pd.DataFrame) -> pd.DataFrame:
    """Return mean, median, standard deviation, and IQR for every numeric column."""
    numeric_df = data.select_dtypes(include=np.number)
    summary = pd.DataFrame(index=numeric_df.columns)
    summary["mean"] = numeric_df.mean()
    summary["median"] = numeric_df.median()
    summary["std"] = numeric_df.std()

    q1 = np.percentile(numeric_df.dropna().to_numpy(), 25, axis=0)
    q3 = np.percentile(numeric_df.dropna().to_numpy(), 75, axis=0)
    summary["iqr"] = q3 - q1
    return summary

numeric_summary = build_numeric_summary(df)
numeric_summary

,mean,median,std,iqr
track_popularity,42.280642,45.000000,24.834982,38.00000
danceability,0.655129,0.672000,0.145247,0.19800
energy,0.698848,0.722000,0.180729,0.25900
key,5.379089,6.000000,3.611675,7.00000
loudness,-6.727628,-6.170000,2.992946,3.53600
mode,0.564820,1.000000,0.495788,1.00000
speechiness,0.107410,0.062700,0.101571,0.09090
acousticness,0.175320,0.080700,0.219153,0.23980
instrumentalness,0.084246,0.000016,0.223835,0.00465
liveness,0.190358,0.127000,0.154677,0.15630


## Stage 2 Genre and Popularity Analysis

In this stage, I compare genres based on popularity and audio features. This helps us understand whether different playlist genres have different musical patterns.

In [6]:
genre_stats = df.groupby("playlist_genre")[["track_popularity", "danceability", "energy", "valence"]].agg(["mean", "std", "median"])
genre_stats

track_popularity                   danceability                     energy                    valence  \
                           mean        std median         mean       std median      mean       std median      mean   
playlist_genre                                                                                                         
edm                   34.561660  22.590491   36.0     0.654409  0.123619  0.658  0.803937  0.137765  0.831  0.401555   
latin                 46.314444  25.142676   50.0     0.713812  0.114819  0.729  0.710585  0.151479  0.732  0.607518   
pop                   47.744870  25.158331   52.0     0.639302  0.128221  0.652  0.701028  0.171084  0.727  0.503521   
r&b                   40.480583  25.768116   43.0     0.672034  0.137582  0.691  0.590671  0.179067  0.595  0.533784   
rap                   43.215454  23.302085   47.0     0.718353  0.136452  0.737  0.650708  0.170340  0.665  0.505090   
rock                  41.971845  24.892926   47.0     0.520037  0.140248  0.523  0.733019  0.194851  0.776  0.538093   

                                 
                     std median  
playlist_genre                   
edm             0.226352  0.371  
latin           0.222016  0.632  
pop             0.220472  0.500  
r&b             0.225664  0.544  
rap             0.224663  0.517  
rock            0.229907  0.532

In [8]:
popularity_variance_by_genre = df.groupby("playlist_genre")["track_popularity"].var().sort_values(ascending=False)
highest_variance_genre = popularity_variance_by_genre.index[0]

print("Track popularity variance by genre:")
display(popularity_variance_by_genre)
print(f"Genre with highest variance in track popularity: {highest_variance_genre}")

Track popularity variance by genre:


playlist_genre
r&b      663.995786
pop      632.941617
latin    632.154177
rock     619.657789
rap      542.987166
edm      510.330306
Name: track_popularity, dtype: float64

Genre with highest variance in track popularity: r&b


**Interpretation:** The genre with the highest popularity variance contains both very popular and less popular tracks. From a recommendation business perspective, this means the genre is risky but interesting: recommending only the genre label is not enough, because the quality or popularity inside that genre is very mixed. A recommendation system should use more detailed audio and user-behavior features, not just genre.

In [10]:
top_10_artists_by_volume = df["track_artist"].value_counts().head(10)

top_artist_stats = (
    df[df["track_artist"].isin(top_10_artists_by_volume.index)]
    .groupby("track_artist")
    .agg(track_count=("track_id", "count"), mean_popularity=("track_popularity", "mean"))
    .sort_values("track_count", ascending=False)
)

artist_with_highest_mean_popularity = top_artist_stats["mean_popularity"].idxmax()

print("Top 10 artists by number of tracks:")
display(top_artist_stats)
print(f"Among those 10, artist with highest mean popularity: {artist_with_highest_mean_popularity}")

volume_quality_corr = top_artist_stats["track_count"].corr(top_artist_stats["mean_popularity"])
print(f"Correlation between track volume and mean popularity among top 10 artists: {volume_quality_corr:.3f}")

Top 10 artists by number of tracks:


,track_count,mean_popularity
track_artist,,
Martin Garrix,159,47.754717
Queen,135,43.318519
The Chainsmokers,119,57.126050
David Guetta,105,53.695238
Don Omar,102,41.950980
Drake,98,45.581633
Dimitri Vegas & Like Mike,92,43.260870
Calvin Harris,91,61.813187
Hardwell,83,39.421687


Among those 10, artist with highest mean popularity: Kygo
Correlation between track volume and mean popularity among top 10 artists: -0.192


**Does volume correlate with quality/popularity?** I treat average `track_popularity` as a rough proxy for popularity, not true quality. The correlation above shows whether artists with more tracks in the dataset also tend to have higher average popularity. If the value is close to 0, volume does not strongly explain popularity. If it is positive, more represented artists are also more popular on average; if negative, having many tracks does not mean higher popularity.

In [11]:
hit_dance_tracks = df[
    (df["track_popularity"] > 70) &
    (df["danceability"] > 0.7) &
    (df["energy"] > 0.6) &
    (df["duration_ms"] < 240000)
].copy()

print("Number of tracks passing all filters:", len(hit_dance_tracks))

if len(hit_dance_tracks) > 0:
    genre_counts = hit_dance_tracks["playlist_genre"].value_counts()
    dominant_genre = genre_counts.idxmax()
    print("Dominant genre:", dominant_genre)
    display(genre_counts)
else:
    print("No tracks passed the filter.")

Number of tracks passing all filters: 1072
Dominant genre: latin


playlist_genre
latin    382
pop      264
rap      173
r&b      127
edm       93
rock      33
Name: count, dtype: int64

## Stage 3 NumPy Analysis

Now I use NumPy directly for feature scaling, correlation, and boolean masking. This part is useful because many ML preprocessing steps are basically array operations.

In [12]:
audio_features = [
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo"
]

X = df[audio_features].to_numpy(dtype=float)

# Vectorized min-max scaling, no loops and no sklearn.
X_min = X.min(axis=0)
X_max = X.max(axis=0)
X_scaled = (X - X_min) / (X_max - X_min)

scaled_preview = pd.DataFrame(X_scaled, columns=audio_features).head()
scaled_preview

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0.760936,0.915985,0.918090,0.063508,0.102616,0.000000,0.065562,0.522704,0.509673
1,0.738555,0.814968,0.869162,0.040632,0.072837,0.004235,0.358434,0.699294,0.417524
2,0.686673,0.930988,0.901368,0.080828,0.079879,0.000023,0.110442,0.618567,0.517908
3,0.730417,0.929988,0.894118,0.111111,0.028873,0.000009,0.204819,0.279516,0.509338
4,0.661241,0.832971,0.875385,0.039107,0.080785,0.000000,0.083635,0.731584,0.517775


In [13]:
corr_matrix = np.corrcoef(X, rowvar=False)
corr_df = pd.DataFrame(corr_matrix, index=audio_features, columns=audio_features)

# Ignore diagonal values because every feature perfectly correlates with itself.
mask = np.eye(len(audio_features), dtype=bool)
corr_no_diag = corr_matrix.copy()
corr_no_diag[mask] = np.nan

max_pos_idx = np.unravel_index(np.nanargmax(corr_no_diag), corr_no_diag.shape)
max_neg_idx = np.unravel_index(np.nanargmin(corr_no_diag), corr_no_diag.shape)

max_positive_pair = (audio_features[max_pos_idx[0]], audio_features[max_pos_idx[1]], corr_matrix[max_pos_idx])
most_negative_pair = (audio_features[max_neg_idx[0]], audio_features[max_neg_idx[1]], corr_matrix[max_neg_idx])

print("Correlation matrix:")
display(corr_df)

print("Highest positive correlation pair:", max_positive_pair)
print("Most negative correlation pair:", most_negative_pair)

Correlation matrix:


,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
danceability,1.000000,-0.087065,0.024051,0.182490,-0.023507,-0.008672,-0.125113,0.331098,-0.185737
energy,-0.087065,1.000000,0.676362,-0.032796,-0.537739,0.028811,0.162031,0.151933,0.148851
loudness,0.024051,0.676362,1.000000,0.010680,-0.360089,-0.152989,0.078444,0.053410,0.093989
speechiness,0.182490,-0.032796,0.010680,1.000000,0.027072,-0.103527,0.055927,0.063282,0.044464
acousticness,-0.023507,-0.537739,-0.360089,0.027072,1.000000,-0.004081,-0.076538,-0.015942,-0.111674
instrumentalness,-0.008672,0.028811,-0.152989,-0.103527,-0.004081,1.000000,-0.005449,-0.173279,0.022131
liveness,-0.125113,0.162031,0.078444,0.055927,-0.076538,-0.005449,1.000000,-0.021803,0.020433
valence,0.331098,0.151933,0.053410,0.063282,-0.015942,-0.173279,-0.021803,1.000000,-0.025909
tempo,-0.185737,0.148851,0.093989,0.044464,-0.111674,0.022131,0.020433,-0.025909,1.000000


Highest positive correlation pair: ('energy', 'loudness', np.float64(0.6763615277715673))
Most negative correlation pair: ('energy', 'acousticness', np.float64(-0.537739449377393))


**Musical interpretation:** The strongest positive pair means those two audio characteristics tend to increase together. For example, if `energy` and `loudness` are highly positive, louder songs are also usually more energetic. The most negative pair means one characteristic tends to decrease when the other increases. For example, if `energy` and `acousticness` are strongly negative, highly energetic songs are usually less acoustic or more electronically produced.

In [14]:
energy_array = df["energy"].to_numpy(dtype=float)
popularity_array = df["track_popularity"].to_numpy(dtype=float)

energy_threshold = energy_array.mean() + energy_array.std()
high_energy_mask = energy_array > energy_threshold

high_energy_popularity_mean = popularity_array[high_energy_mask].mean()
overall_popularity_mean = popularity_array.mean()

print(f"Energy threshold (mean + std): {energy_threshold:.4f}")
print("Number of high-energy tracks:", high_energy_mask.sum())
print(f"Mean popularity of high-energy subset: {high_energy_popularity_mean:.2f}")
print(f"Overall mean popularity: {overall_popularity_mean:.2f}")
print(f"Difference: {high_energy_popularity_mean - overall_popularity_mean:.2f}")

Energy threshold (mean + std): 0.8796
Number of high-energy tracks: 5508
Mean popularity of high-energy subset: 36.29
Overall mean popularity: 42.28
Difference: -5.99


## Stage 4 Documentation and AI Tool Reflection

I used an AI coding assistant style prompt to improve docstrings for two utility functions. I did not blindly accept everything. I kept the parts that explained parameters and returns clearly, then shortened the wording so the notebook stays readable.

### Prompt used for docstrings

> Write concise but clear Python docstrings for two functions in a Spotify EDA notebook. The first function builds a numeric summary table with mean, median, standard deviation, and manually computed IQR. The second function computes an artist audio fingerprint as the mean vector of nine Spotify audio features. Include parameters, return values, and one sentence about why each function is useful.

### Evaluation of AI output

The AI output was useful, but slightly too formal and long. I modified the wording to make it shorter and more natural. I kept the parameter and return explanations because they make the functions easier to reuse.

## Bonus 1 Artist Audio Fingerprint

This function represents each artist using the average of their nine audio features. Then I compare artists using cosine similarity from the mathematical definition, without sklearn or scipy.

In [15]:
def artist_fingerprint(data: pd.DataFrame, artist_name: str) -> np.ndarray:
    """Return an artist's average nine-feature audio vector as a NumPy array.

    Parameters
    ----------
    data : pd.DataFrame
        Spotify dataset containing artist names and audio feature columns.
    artist_name : str
        Artist name to search for. Matching is case-insensitive.

    Returns
    -------
    np.ndarray
        Mean audio feature vector for the selected artist.

    Raises
    ------
    ValueError
        If the artist is not found in the dataset.
    """
    mask = data["track_artist"].str.lower() == artist_name.lower()
    if not mask.any():
        raise ValueError(f"Artist not found: {artist_name}")
    return data.loc[mask, audio_features].mean().to_numpy(dtype=float)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity using the formula dot(A, B) / (||A|| * ||B||)."""
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return np.nan
    return float(np.dot(a, b) / denominator)

# Pick artists automatically from the dataset so the code runs safely.
artists_by_genre = (
    df.groupby(["playlist_genre", "track_artist"])
    .size()
    .reset_index(name="count")
    .sort_values(["playlist_genre", "count"], ascending=[True, False])
)

candidate_pairs = []

# Same-genre pairs: take top two artists from genres that have at least two artists.
for genre, group in artists_by_genre.groupby("playlist_genre"):
    names = group["track_artist"].head(2).tolist()
    if len(names) == 2:
        candidate_pairs.append((names[0], names[1], f"same genre: {genre}"))
    if len(candidate_pairs) >= 2:
        break

# Different-genre pair: take top artist from first two genres.
top_by_genre = artists_by_genre.groupby("playlist_genre").head(1)
if top_by_genre["playlist_genre"].nunique() >= 2:
    first_two = top_by_genre.drop_duplicates("playlist_genre").head(2)
    candidate_pairs.append((first_two.iloc[0]["track_artist"], first_two.iloc[1]["track_artist"], "different genres"))

pair_results = []
for artist_a, artist_b, pair_type in candidate_pairs[:3]:
    fp_a = artist_fingerprint(df, artist_a)
    fp_b = artist_fingerprint(df, artist_b)
    sim = cosine_similarity(fp_a, fp_b)
    pair_results.append({"artist_a": artist_a, "artist_b": artist_b, "pair_type": pair_type, "cosine_similarity": sim})

pair_results_df = pd.DataFrame(pair_results)
pair_results_df

,artist_a,artist_b,pair_type,cosine_similarity
0,Martin Garrix,Dimitri Vegas & Like Mike,same genre: edm,0.999946
1,Don Omar,Daddy Yankee,same genre: latin,0.999971
2,Martin Garrix,Don Omar,different genres,0.999981


**Interpretation:** Cosine similarity close to 1 means the artists have very similar audio profiles based on these nine features. Same-genre pairs usually should have higher similarity, but the result may not always be perfect because genre labels are broad and artists can have diverse songs.

## Bonus 2 Genre Cluster Profile

Here I create one audio centroid per genre, then calculate the full pairwise Euclidean distance matrix using NumPy broadcasting. Smaller distance means more similar audio profile.

In [ ]:
genre_centroids_df = df.groupby("playlist_genre")[audio_features].mean()
genres = genre_centroids_df.index.to_numpy()
centroids = genre_centroids_df.to_numpy(dtype=float)

# vectorized pairwise Euclidean distance matrix.
diff = centroids[:, np.newaxis, :] - centroids[np.newaxis, :, :]
distance_matrix = np.sqrt(np.sum(diff ** 2, axis=2))
distance_df = pd.DataFrame(distance_matrix, index=genres, columns=genres)

distance_df

,edm,latin,pop,r&b,rap,rock
edm,0.000000,7.238056,5.238140,11.931213,5.494725,2.333361
latin,7.238056,0.000000,2.030702,4.774306,2.094220,6.486733
pop,5.238140,2.030702,0.000000,6.701257,0.749149,4.506707
r&b,11.931213,4.774306,6.701257,0.000000,6.481642,10.839213
rap,5.494725,2.094220,0.749149,6.481642,0.000000,4.449504
rock,2.333361,6.486733,4.506707,10.839213,4.449504,0.000000


In [17]:
distance_no_diag = distance_matrix.copy()
np.fill_diagonal(distance_no_diag, np.nan)

most_similar_idx = np.unravel_index(np.nanargmin(distance_no_diag), distance_no_diag.shape)
most_different_idx = np.unravel_index(np.nanargmax(distance_no_diag), distance_no_diag.shape)

most_similar_genres = (genres[most_similar_idx[0]], genres[most_similar_idx[1]], distance_matrix[most_similar_idx])
most_different_genres = (genres[most_different_idx[0]], genres[most_different_idx[1]], distance_matrix[most_different_idx])

print("Most similar genres:", most_similar_genres)
print("Most different genres:", most_different_genres)

Most similar genres: ('pop', 'rap', np.float64(0.7491492326854329))
Most different genres: ('edm', 'r&b', np.float64(11.931213165780555))


In [18]:
def recommend_similar_genre(genre: str, top_k: int = 3) -> pd.DataFrame:
    """Return the top-k most similar genres based on the genre distance matrix."""
    if genre not in distance_df.index:
        available = ", ".join(distance_df.index.astype(str))
        raise ValueError(f"Genre '{genre}' not found. Available genres: {available}")

    distances = distance_df.loc[genre].drop(index=genre).sort_values().head(top_k)
    return distances.reset_index().rename(columns={"index": "similar_genre", genre: "distance"})

example_genre = genres[0]
print("Example genre:", example_genre)
recommend_similar_genre(example_genre, top_k=3)

Example genre: edm


,similar_genre,distance
0,rock,2.333361
1,pop,5.238140
2,rap,5.494725


**Interpretation:** If the closest genres are musically related, the result matches intuition. If not, it may be because the comparison only uses averaged audio features, not lyrics, culture, subgenre, or listener context.

## Bonus 3 Reusable Analysis Class

Finally, I refactor the core analysis into a reusable `SpotifyAnalyzer` class. The code is designed to run from a single Colab cell with no extra setup besides internet access.

In [19]:
from typing import Dict, List, Tuple, Any

class SpotifyAnalyzer:
    """Reusable analyzer for the Spotify Songs EDA task."""

    def __init__(self, url: str) -> None:
        """Load the Spotify dataset from a CSV URL and store cleaned working data."""
        self.url = url
        self.df_raw = pd.read_csv(url)
        self.df = self._clean_data(self.df_raw)
        self.audio_features = [
            "danceability", "energy", "loudness", "speechiness", "acousticness",
            "instrumentalness", "liveness", "valence", "tempo"
        ]

    def _clean_data(self, data: pd.DataFrame) -> pd.DataFrame:
        """Drop high-missing columns and remove duplicate records."""
        cleaned = data.copy()
        high_missing_cols = cleaned.columns[cleaned.isna().mean() > 0.20].tolist()
        cleaned = cleaned.drop(columns=high_missing_cols)
        cleaned = cleaned.drop_duplicates()
        if {"track_id", "playlist_id"}.issubset(cleaned.columns):
            cleaned = cleaned.drop_duplicates(subset=["track_id", "playlist_id"], keep="first")
        return cleaned

    def initial_inspection(self) -> Dict[str, Any]:
        """Return shape, missing values, and duplicate counts for quick inspection."""
        return {
            "shape_raw": self.df_raw.shape,
            "shape_clean": self.df.shape,
            "missing_values": self.df.isna().sum().to_dict(),
            "exact_duplicates_clean": int(self.df.duplicated().sum())
        }

    def numeric_summary(self) -> pd.DataFrame:
        """Build a numeric summary table with mean, median, std, and manual NumPy IQR."""
        return build_numeric_summary(self.df)

    def genre_analysis(self) -> Dict[str, Any]:
        """Analyze genre-level popularity and audio feature statistics."""
        genre_stats_local = self.df.groupby("playlist_genre")[["track_popularity", "danceability", "energy", "valence"]].agg(["mean", "std", "median"])
        variance = self.df.groupby("playlist_genre")["track_popularity"].var().sort_values(ascending=False)
        return {
            "genre_stats": genre_stats_local,
            "highest_popularity_variance_genre": variance.index[0],
            "popularity_variance": variance
        }

    def artist_volume_analysis(self) -> pd.DataFrame:
        """Return track count and mean popularity for the ten most represented artists."""
        top_artists = self.df["track_artist"].value_counts().head(10).index
        return (
            self.df[self.df["track_artist"].isin(top_artists)]
            .groupby("track_artist")
            .agg(track_count=("track_id", "count"), mean_popularity=("track_popularity", "mean"))
            .sort_values("track_count", ascending=False)
        )

    def numpy_audio_analysis(self) -> Dict[str, Any]:
        """Run NumPy scaling, correlation, and high-energy popularity comparison."""
        X_local = self.df[self.audio_features].to_numpy(dtype=float)
        X_scaled_local = (X_local - X_local.min(axis=0)) / (X_local.max(axis=0) - X_local.min(axis=0))
        corr_local = np.corrcoef(X_local, rowvar=False)

        corr_no_diag_local = corr_local.copy()
        np.fill_diagonal(corr_no_diag_local, np.nan)
        max_pos = np.unravel_index(np.nanargmax(corr_no_diag_local), corr_no_diag_local.shape)
        max_neg = np.unravel_index(np.nanargmin(corr_no_diag_local), corr_no_diag_local.shape)

        energy_local = self.df["energy"].to_numpy(dtype=float)
        pop_local = self.df["track_popularity"].to_numpy(dtype=float)
        high_energy_mask_local = energy_local > energy_local.mean() + energy_local.std()

        return {
            "scaled_shape": X_scaled_local.shape,
            "highest_positive_correlation_pair": (
                self.audio_features[max_pos[0]], self.audio_features[max_pos[1]], float(corr_local[max_pos])
            ),
            "most_negative_correlation_pair": (
                self.audio_features[max_neg[0]], self.audio_features[max_neg[1]], float(corr_local[max_neg])
            ),
            "high_energy_mean_popularity": float(pop_local[high_energy_mask_local].mean()),
            "overall_mean_popularity": float(pop_local.mean())
        }

    def generate_report(self) -> Dict[str, Any]:
        """Return key EDA insights in a JSON-serializable dictionary."""
        genre_info = self.genre_analysis()
        artist_info = self.artist_volume_analysis()
        numpy_info = self.numpy_audio_analysis()

        return {
            "rows_after_cleaning": int(self.df.shape[0]),
            "columns_after_cleaning": int(self.df.shape[1]),
            "highest_popularity_variance_genre": str(genre_info["highest_popularity_variance_genre"]),
            "top_volume_artist": str(artist_info.index[0]),
            "highest_mean_popularity_among_top_10_artists": str(artist_info["mean_popularity"].idxmax()),
            "highest_positive_correlation_pair": numpy_info["highest_positive_correlation_pair"],
            "most_negative_correlation_pair": numpy_info["most_negative_correlation_pair"],
            "high_energy_mean_popularity": round(numpy_info["high_energy_mean_popularity"], 2),
            "overall_mean_popularity": round(numpy_info["overall_mean_popularity"], 2)
        }

analyzer = SpotifyAnalyzer(DATA_URL)
report = analyzer.generate_report()
report

{'rows_after_cleaning': 32251,
 'columns_after_cleaning': 23,
 'highest_popularity_variance_genre': 'r&b',
 'top_volume_artist': 'Martin Garrix',
 'highest_mean_popularity_among_top_10_artists': 'Kygo',
 'highest_positive_correlation_pair': ('energy',
  'loudness',
  0.6763615277715673),
 'most_negative_correlation_pair': ('energy',
  'acousticness',
  -0.537739449377393),
 'high_energy_mean_popularity': 36.29,
 'overall_mean_popularity': 42.28}

### AI documentation prompt for the class

> I have a Python class called `SpotifyAnalyzer` for a Spotify EDA notebook. It loads a CSV URL, cleans missing and duplicate data, summarizes numeric features, analyzes genres, analyzes artist volume, runs NumPy audio analysis, and returns a report dictionary. Write concise docstrings with type-aware explanations for every method.

### Evaluation of AI output

The AI-generated docstrings were structurally good, but a little too verbose. I shortened them so the class stays readable while still explaining what each method does.

## Final Three Key Insights

1. Some genres have much more mixed popularity than others. This means a recommendation system should not rely only on genre labels, because one genre can contain both very popular and less popular songs.

2. Having many tracks in the dataset does not automatically mean an artist has the highest average popularity. Track volume and popularity are related only if the calculated correlation supports it.

3. Audio features are connected in meaningful ways. For example, energetic songs often share patterns with loudness or lower acousticness, which makes sense musically because high-energy tracks are often produced to sound stronger and more intense.